#### Objetivo: Apresentar as métricas dos modelos dado os hiperparâmetros p,x_lag,real_degree,img_degree

In [2]:
# ---------------------
# Incluindo Bibliotecas
# ---------------------
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VARMAX
from statsmodels.iolib.smpickle import load_pickle
from pathlib import Path
from fpdf import FPDF


##### Lendo base de dados

In [3]:
def get_result_file_name(p:int,xl:int,r:int,i:int) -> str:
    fname = f"larger-dataset-test-results-p{p}-xl{xl}-r{r}-i{i}.csv"
    return fname

In [4]:
project_path = Path.cwd().parent
csv_path = f"../results/{get_result_file_name(3,3,3,3)}"
df_res = pd.read_csv(csv_path)
data = pd.read_excel("../data/dadosIniciais.xlsx")

In [5]:
df_input = pd.DataFrame()
df_input["Xreal"] = data.iloc[:,0]
df_input["Ximg"] = data.iloc[:,1]

In [6]:
df_output = pd.DataFrame()
df_output["Yreal"] = data.iloc[:,2]
df_output["Yimg"] = data.iloc[:,3]

In [7]:
df_concat = pd.concat([df_output,df_input],axis=1)
best_yr_param = pd.Series(df_res.sort_values(by="Yreal_RMSE").iloc[0])
best_yi_param = pd.Series(df_res.sort_values(by="Yimg_RMSE").iloc[0])

#### Visualizando Dados

In [8]:
# -----------------------------
# Função de Impressão dos Dados
# -----------------------------
def view_result(data,meta=None,endog_cols=('Yreal', 'Yimg')):
    """
    Imprime na tela as métricas calculadas de acordo com a previsão do modelo.
    Aceita `data` como dicionário ou DataFrame (métricas nas linhas, colunas endog nas colunas).
    `meta` é obrigatório quando `data` for um DataFrame: dict com p, aic, bic, rmse_mean, x_lag, exog_degree.
    """
    metric_labels = [
        ('RMSE',  'RMSE'),
        ('MAE',   'MAE'),
        ('R2',    'R²'),
        ('AdjR2', 'R² Ajust'),
        ('STD',   'STD')]

    W_METRIC = 10
    W_VAL    = 10

    def sep(l, m, r, n_cols):
        block = '─' * (W_METRIC + W_VAL + 3)
        return l + m.join([block] * n_cols) + r

    def row_str(pairs):
        cells = [f' {lbl:<{W_METRIC}} {val:>{W_VAL}} ' for lbl, val in pairs]
        return '│' + '│'.join(cells) + '│'

    is_df = isinstance(data, pd.DataFrame)

    def get_meta(key):
        return meta[key] if is_df else data[key]

    def get_metric(col, key):
        return data.at[key, col] if is_df else data[col][key]

    print(f'Lag={get_meta("p")}')
    print(f'  AIC={get_meta("aic"):.2f} | BIC={get_meta("bic"):.2f} | RMSE Mean={get_meta("rmse_mean"):.4f}')
    print(f'  X Lag={get_meta("x_lag")} | Exog Degree={get_meta("exog_degree")}')

    print(sep('  ┌', '┬', '┐', len(endog_cols)))
    print('  ' + row_str([(col, '') for col in endog_cols]))
    print(sep('  ├', '┼', '┤', len(endog_cols)))

    for key, lbl in metric_labels:
        pairs = []
        for col in endog_cols:
            val    = get_metric(col, key)
            v_str  = f'{val:+.4f}' if key in ('R2', 'AdjR2') else f'{val:.4f}'
            pairs.append((lbl, v_str))
        print('  ' + row_str(pairs))

    print(sep('  └', '┴', '┘', len(endog_cols)))

In [9]:
# ------------------------------------------------
# Função de Recuperação de Dados para Visualização
# ------------------------------------------------
def row_to_metrics_df(row:pd.Series,endog_cols:list=('Yreal', 'Yimg')) -> tuple[pd.DataFrame, dict]:
    """
    Reconstrói o df_metrics e meta a partir de uma linha do DataFrame de resultados.
    """
    metric_keys = ['MSE', 'RMSE', 'MAE', 'R2', 'AdjR2', 'STD']
    meta_keys   = ['p', 'x_lag', 'exog_degree', 'trend', 'aic', 'bic', 'rmse_mean']

    records = {}
    for col in endog_cols:
        records[col] = {m: row[f'{col}_{m}'] for m in metric_keys}

    df_metrics = pd.DataFrame(records)
    meta       = {k: row[k] for k in meta_keys}

    return df_metrics, meta

In [10]:
metrics, meta = row_to_metrics_df(best_yr_param)
view_result(metrics,meta=meta)

Lag=2
  AIC=919964.80 | BIC=920483.15 | RMSE Mean=0.5326
  X Lag=2 | Exog Degree=(r:3,i:3)
  ┌───────────────────────┬───────────────────────┐
  │ Yreal                 │ Yimg                  │
  ├───────────────────────┼───────────────────────┤
  │ RMSE           0.5321 │ RMSE           0.5332 │
  │ MAE            0.4400 │ MAE            0.4412 │
  │ R²            +0.9983 │ R²            +0.9983 │
  │ R² Ajust      +0.9983 │ R² Ajust      +0.9983 │
  │ STD            0.5321 │ STD            0.5331 │
  └───────────────────────┴───────────────────────┘


In [11]:
metrics, meta = row_to_metrics_df(best_yi_param)
view_result(metrics,meta=meta)

Lag=1
  AIC=917792.51 | BIC=918014.66 | RMSE Mean=0.5326
  X Lag=0 | Exog Degree=(r:3,i:3)
  ┌───────────────────────┬───────────────────────┐
  │ Yreal                 │ Yimg                  │
  ├───────────────────────┼───────────────────────┤
  │ RMSE           0.5321 │ RMSE           0.5331 │
  │ MAE            0.4401 │ MAE            0.4411 │
  │ R²            +0.9983 │ R²            +0.9983 │
  │ R² Ajust      +0.9983 │ R² Ajust      +0.9983 │
  │ STD            0.5321 │ STD            0.5331 │
  └───────────────────────┴───────────────────────┘


#### Carregando Modelo

In [220]:
# ---------------------------------------------------
# Modificiando DataFrame (Replicando e Aleatorizando)
# ---------------------------------------------------
def repeat_data(df:pd.DataFrame,n:int,random:bool=False,rdm_state:int=42):
    """Replica os dados do dataframe n vezes e retorna um novo dataframe"""
    df_temp = df.loc[df.index.repeat(n)].reset_index(drop=True)
    if(random):
        df_temp = df_temp.sample(frac=1,random_state=rdm_state).reset_index(drop=True)
    return df_temp 

In [206]:
# ---------------------------------------
# Função de Criação Automática de Lag (m)
# ---------------------------------------
def select_auto_lag(df_exog,exog_cols,max_x_lag=2,inplace=True) -> pd.DataFrame:
  """
  Aplica os lags desejado às exógenas da base de dados original e retorna o dataframe com as novas colunas
  """
  new_exog_cols = list(exog_cols)

  for lag in range(1, max_x_lag + 1):
    for col in exog_cols:
      lag_cols_name = f'{col}_lag{lag}'
      df_exog[lag_cols_name] = df_exog[col].shift(lag)
      new_exog_cols.append(lag_cols_name)

  lag_cols = [f'{col}_lag{lag}' for lag in range(1, max_x_lag+1) for col in exog_cols]
  rows_to_drop = df_exog.index[df_exog[lag_cols].isna().any(axis=1)]

  df_exog = df_exog.drop(index=rows_to_drop)
  return df_exog,new_exog_cols

In [207]:
# -------------------------------------------------
# Função de Seleção de grau de Polinômio de Lag (m)
# -------------------------------------------------
def select_auto_polinomial_degree(df_exog,exog_cols,degree_type,max_degree=1) -> pd.DataFrame:
  """
  Aplica os graus desejado às exógenas da base de dados original, gerando o polinômio completo para cada coluna
  das exógenas (originais e sem lags) e retorna o dataframe com as novas colunas
  """
  new_exog_cols = list(exog_cols)
  new_columns = {}

  if degree_type in ['real','r','REAL','Real']:
    degree_type = 'real'
  elif degree_type in ['img','i','IMG','Img']:
    degree_type = 'img'
  else:
    raise ValueError("degree_type must be 'real' or 'img'")

  # Aplicando graus para série de dados reais
  if max_degree >= 2:
    for degree in range(2, max_degree + 1):
      columns = [col for col in exog_cols if degree_type in col]
      for col in columns:
        degree_col_name = f'{col}_deg{degree}'
        new_columns[degree_col_name] = df_exog[col] ** degree
        new_exog_cols.append(degree_col_name)

  if new_columns:
    df_exog = pd.concat([df_exog,pd.DataFrame(new_columns,index=df_exog.index)],axis=1)

  return df_exog,new_exog_cols

In [208]:
# --------------------------------------------
# Função de Seleção de Teste de Lag (m)
# --------------------------------------------
def calc_train_sect(data: pd.DataFrame, train_frac: float, val_frac: float):
    n = len(data)
    train_end = int(n * train_frac)
    val_end   = int(n * (train_frac + val_frac))
    return train_end, val_end

In [209]:
# --------------------------------------------
# Geração do DataFrame Customizado para Treino
# --------------------------------------------
def get_df_train(df_train:pd.DataFrame,exog_lag:int,exog_real_degree:int,exog_img_degree:int,exog_cols=("Xreal","Ximg")):
  df_temp = df_train.copy()
  current_exog_cols = list(exog_cols)
  
  if (exog_lag > 0):
    df_temp,current_exog_cols = select_auto_lag(df_train,exog_cols,max_x_lag=exog_lag)
  
  if (exog_real_degree >= 2):
    df_temp,current_exog_cols = select_auto_polinomial_degree(df_temp,current_exog_cols,'r',max_degree=exog_real_degree)

  if (exog_img_degree >= 2):
    df_temp,current_exog_cols = select_auto_polinomial_degree(df_temp,current_exog_cols,'i',max_degree=exog_img_degree)
  
  return df_temp,current_exog_cols

# ----------------------------------------
# Treinamento do Modelo VARMAX customizado
# ----------------------------------------
def custom_train_varmax(
    df_train:pd.DataFrame,
    endog_lag,
    exog_lag,
    exog_img_degree,
    exog_real_degree,
    train_frac=0.6,
    val_frac=0.2,
    endog_cols=("Yreal","Yimg"),
    exog_cols=("Xreal", "Ximg")) -> VARMAX:
  """
  Treina o modelo VARMAX com base nos parâmetros fornecidos de lags para endógenas e exógenas e também
  com os graus dos polinômios de exógenas (real e imaginária)
  """
  train_end,val_end = calc_train_sect(df_train,train_frac,val_frac)
  df_temp,current_exog_cols = get_df_train(df_train,exog_lag,exog_real_degree,exog_img_degree,exog_cols=exog_cols)

  train = df_temp.iloc[:train_end].copy()
  val = df_temp.iloc[train_end:val_end].copy()
  test = df_temp.iloc[val_end:].copy()

  model = VARMAX(
          train[list(endog_cols)],
          exog=train[current_exog_cols],
          order=(endog_lag, 0),
          trend='c')

  res = model.fit(disp=False,maxiter=500,method="powell")

  return res,df_temp,current_exog_cols

In [ ]:
# Treinando/Carregando modelo com os seguintes parâmetros (tempo de aproximadamente 53 min)
endog_cols = ["Yreal","Yimg"]
exog_cols =  ["Xreal","Ximg"]

params = {'p':1,'x_lag':1,'r':1,'i':2}
base_dados = {'n_mul': 1, 'random': False, 'seed': 42}
report_name_flag = "(original)"
base_dados['n'] = len(df_concat)*base_dados['n_mul']

model_pkl_path = Path(f"{project_path}/models/varmax-p{params['p']}-xl{params['x_lag']}-r{params['r']}-i{params['i']}.pkl")
final_df_train = repeat_data(df_concat,base_dados['n_mul'],random=base_dados['random'],rdm_state=base_dados['seed'])

if not(model_pkl_path.exists()):
    print("[File not found] Training model...")
    final_res,df_train_test,current_exog_cols = custom_train_varmax(
        df_train=final_df_train,
        endog_lag=params['p'],
        exog_lag=params['x_lag'],
        exog_img_degree=params['i'],
        exog_real_degree=params['r'])

    final_res.save(f"models/varmax-p{params['p']}-xl{params['x_lag']}-r{params['r']}-i{params['i']}.pkl")
    trained = True

else:
    df_train_test,current_exog_cols = get_df_train(final_df_train,params['x_lag'],params['r'],params['i'])
    final_res = load_pickle(str(model_pkl_path))
    print(model_pkl_path)



/home/mgs/Documents/FACOM/Projects/projeto-dpd/masters-varmax/models/varmax-p1-xl1-r1-i2.pkl


#### Gerando PDF

In [211]:
# ------------------------
# Carregar métricas do CSV
# ------------------------
def load_metrics(csv_path: str, params: dict) -> pd.Series:
    df = pd.read_csv(csv_path)
    exog_degree_str = f"(r:{params['r']},i:{params['i']})"
    mask = (
        (df['p']           == params['p'])        &
        (df['x_lag']       == params['x_lag'])    &
        (df['exog_degree'] == exog_degree_str)
    )
    result = df[mask]
    if result.empty:
        raise ValueError(f"Nenhuma métrica encontrada para os parâmetros: {params}\n"
                         f"exog_degree buscado: '{exog_degree_str}'")
    return result.iloc[0]

metrics = load_metrics(csv_path, params)
print(metrics)

Unnamed: 0                62
rmse_mean           0.532619
aic            919964.798212
bic            920483.152712
p                          2
x_lag                      2
exog_degree        (r:3,i:3)
trend                      c
Yreal_MSE           0.283116
Yreal_RMSE          0.532087
Yreal_MAE           0.439994
Yreal_R2            0.998332
Yreal_AdjR2         0.998332
Yreal_STD           0.532087
Yimg_MSE            0.284251
Yimg_RMSE           0.533152
Yimg_MAE            0.441164
Yimg_R2             0.998321
Yimg_AdjR2          0.998321
Yimg_STD            0.533149
Name: 62, dtype: object


In [1]:
# -------------------------------------------------------------
# Função auxiliar: gráfico de e magnitude constelação em buffer
# -------------------------------------------------------------
def constellation_buf(data, title: str, endog_cols: tuple, figsize=(6,6)) -> io.BytesIO:
    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(data[endog_cols[0]], data[endog_cols[1]], s=1, color='blue')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("In-phase (I) [Y-Real]")
    ax.set_ylabel("Quadrature (Q) [Y-Img]")
    ax.grid(True)
    ax.axhline(0, color='black', lw=1)
    ax.axvline(0, color='black', lw=1)
    ax.set_aspect('equal')
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=96, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return buf

def magnitude_buf(df_input, forecast, title, endog_cols, exog_cols=('Xreal','Ximg')):
    fig, ax = plt.subplots()
    mag_input = np.sqrt(df_input[exog_cols[0]]**2 + df_input[exog_cols[1]]**2)
    mag_output = np.sqrt(forecast[endog_cols[0]]**2 + forecast[endog_cols[1]]**2)
    mag_input_db  = 20 * np.log10(mag_input)
    mag_output_db = 20 * np.log10(mag_output)
    ax.plot(mag_input_db, mag_output_db, marker='o')
    ax.set_xlabel("MAG Input [dB]")
    ax.set_ylabel("MAG Output Forecast [dB]")
    ax.set_title(title)
    ax.grid(True)
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)
    return buf

def comparison_buf(data_orig, data_fc, title_orig: str, title_fc: str, endog_cols: tuple) -> io.BytesIO:
    fig, axes = plt.subplots(2, 1, figsize=(6, 12))
    for ax, data, title in zip(axes, [data_orig, data_fc], [title_orig, title_fc]):
        ax.scatter(data[endog_cols[0]], data[endog_cols[1]], s=1, color='blue')
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("In-phase (I) [Y-Real]")
        ax.set_ylabel("Quadrature (Q) [Y-Img]")
        ax.grid(True)
        ax.axhline(0, color='black', lw=1)
        ax.axvline(0, color='black', lw=1)
        ax.set_aspect('equal')
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=96, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return buf

In [2]:
# -------------
# Classe do PDF
# -------------
class VarmaxReport(FPDF):
    def __init__(self):
        super().__init__()

    def header(self):
        pass  # sem cabeçalho padrão

    def section_title(self, title: str):
        self.set_font("Courier", style="B", size=13)
        self.cell(0, 10, title, ln=True, align="C")
        self.ln(3)

    def field(self, label: str, value: str):
        self.set_font("Courier", style="B", size=10)
        self.cell(70, 6, f"  {label}", ln=False)
        self.set_font("Courier", size=10)
        self.cell(0, 6, f": {value}", ln=True)

    def subsection(self, title: str):
        self.ln(3)
        self.set_font("Courier", style="B", size=11)
        self.cell(0, 8, title, ln=True)

    def add_image_buf(self, buf: io.BytesIO, w=190):
        self.add_page()
        self.image(buf, x=10, y=20, w=w)

    def add_comparison_buf(self, buf: io.BytesIO, w=130):
        self.add_page()
        # Centraliza a imagem verticalmente na página (297mm altura A4)
        self.image(buf, x=(210 - w) / 2, y=10, w=w)

NameError: name 'FPDF' is not defined

In [214]:
# ---------------------------------
# Funções para cada página de texto
# ---------------------------------
def add_cover_page(pdf: VarmaxReport, params: dict, base_dados: dict, final_res):
    pdf.add_page()
    pdf.section_title("RELATORIO DE ANALISE - MODELO VARMAX")

    pdf.subsection("Hiperparametros do Modelo:")
    pdf.field("p   (endog_lag)",        str(params['p']))
    pdf.field("x_lag (exog_lag)",       str(params['x_lag']))
    pdf.field("r   (exog_real_degree)", str(params['r']))
    pdf.field("i   (exog_img_degree)",  str(params['i']))

    pdf.subsection("Equacao do Modelo:")
    pdf.set_font("Courier", size=10)
    pdf.cell(0, 6, f"  Y(t) = AR(1..{params['p']}) + X_real^(1..{params['r']}) + X_img^(1..{params['i']}) com lag(1..{params['x_lag']})", ln=True)

    pdf.subsection("Base de Dados:")
    pdf.field("Tamanho          ",  str(base_dados['n']))
    pdf.field("N° Replicações    ",  str(base_dados['n_mul']))
    pdf.field("Randomizada      ",  str(base_dados['random']))
    pdf.field("Seed             ",  str(base_dados['seed']))
    pdf.set_font("Courier", size=10)

    pdf.subsection("Resumo Estatistico:")
    pdf.set_font("Courier", size=8)
    for line in final_res.summary().tables[0].as_text().splitlines():
        pdf.cell(0, 5, line, ln=True)


def add_metrics_page(pdf: VarmaxReport, params: dict, metrics: pd.Series):
    pdf.add_page()
    pdf.section_title("METRICAS DO MODELO")

    pdf.set_font("Courier", size=10)
    pdf.cell(0, 6, f"  Hiperparametros: p={params['p']}, x_lag={params['x_lag']}, r={params['r']}, i={params['i']}", ln=True)

    pdf.subsection("Metricas Gerais:")
    pdf.field("RMSE Medio", f"{metrics['rmse_mean']:.6f}")
    pdf.field("AIC",        f"{metrics['aic']:.4f}")
    pdf.field("BIC",        f"{metrics['bic']:.4f}")

    pdf.subsection("Componente Real (Yreal):")
    pdf.field("MSE",    f"{metrics['Yreal_MSE']:.6f}")
    pdf.field("RMSE",   f"{metrics['Yreal_RMSE']:.6f}")
    pdf.field("MAE",    f"{metrics['Yreal_MAE']:.6f}")
    pdf.field("R2",     f"{metrics['Yreal_R2']:.6f}")
    pdf.field("Adj R2", f"{metrics['Yreal_AdjR2']:.6f}")
    pdf.field("STD",    f"{metrics['Yreal_STD']:.6f}")

    pdf.subsection("Componente Imaginaria (Yimg):")
    pdf.field("MSE",    f"{metrics['Yimg_MSE']:.6f}")
    pdf.field("RMSE",   f"{metrics['Yimg_RMSE']:.6f}")
    pdf.field("MAE",    f"{metrics['Yimg_MAE']:.6f}")
    pdf.field("R2",     f"{metrics['Yimg_R2']:.6f}")
    pdf.field("Adj R2", f"{metrics['Yimg_AdjR2']:.6f}")
    pdf.field("STD",    f"{metrics['Yimg_STD']:.6f}")

In [ ]:
# --------------------------------
# Preparação dos dados e forecasts
# --------------------------------
# Definindo nome e caminho do PDF
report_dir  = Path(f"{project_path}/reports")
report_dir.mkdir(parents=True, exist_ok=True)  # cria o diretório se não existir
report_name = report_dir / f"relatorio-varmax-p{params['p']}-xl{params['x_lag']}-r{params['r']}-i{params['i']}{report_name_flag}.pdf"

# Separando os splits
train_end, val_end = calc_train_sect(df_train_test, 0.6, 0.2)
train = df_train_test.iloc[:train_end].copy()
val   = df_train_test.iloc[train_end:val_end].copy()
test  = df_train_test.iloc[val_end:].copy()

# Forecasts por split
fc_train = final_res.forecast(steps=len(train[list(endog_cols)]), exog=train[current_exog_cols])
fc_val   = final_res.forecast(steps=len(val[list(endog_cols)]),   exog=val[current_exog_cols])
fc_test  = final_res.forecast(steps=len(test[list(endog_cols)]),  exog=test[current_exog_cols])

# Forecast total
endog_total      = df_train_test[list(["Yreal", "Yimg"])]
exog_total       = df_train_test[list(current_exog_cols)]
fc_total         = final_res.forecast(steps=len(endog_total), exog=exog_total)
fc_total.index   = endog_total.index

In [ ]:
# --------------
# Geração do PDF
# --------------
param_str = f"p:{params['p']}, xl:{params['x_lag']}, r:{params['r']}, i:{params['i']}"
pdf = VarmaxReport()
# Página 1: Capa (texto)
add_cover_page(pdf, params, base_dados, final_res)
# Página 2: Métricas (texto)
add_metrics_page(pdf, params, metrics)

# Treino
pdf.add_image_buf(constellation_buf(train,    "Dados Originais (Treino)",        endog_cols))
pdf.add_image_buf(constellation_buf(fc_train, f"Forecast (Treino)\n{param_str}", endog_cols))
pdf.add_comparison_buf(comparison_buf(train, fc_train, "Dados Originais (Treino)", f"Forecast (Treino)\n{param_str}", endog_cols))
pdf.add_image_buf(magnitude_buf(train, fc_train, f"Mapa de Magnitude (Treino)\n{param_str}", endog_cols))

# Validação
pdf.add_image_buf(constellation_buf(val,    "Dados Originais (Validacao)",        endog_cols))
pdf.add_image_buf(constellation_buf(fc_val, f"Forecast (Validacao)\n{param_str}", endog_cols))
pdf.add_comparison_buf(comparison_buf(val, fc_val, "Dados Originais (Validacao)", f"Forecast (Validacao)\n{param_str}", endog_cols))
pdf.add_image_buf(magnitude_buf(val, fc_val, f"Mapa de Magnitude (Validacao)\n{param_str}", endog_cols))

# Teste
pdf.add_image_buf(constellation_buf(test,    "Dados Originais (Teste)",        endog_cols))
pdf.add_image_buf(constellation_buf(fc_test, f"Forecast (Teste)\n{param_str}", endog_cols))
pdf.add_comparison_buf(comparison_buf(test, fc_test, "Dados Originais (Teste)", f"Forecast (Teste)\n{param_str}", endog_cols))
pdf.add_image_buf(magnitude_buf(test, fc_test, f"Mapa de Magnitude (Teste)\n{param_str}", endog_cols))

# Total
pdf.add_image_buf(constellation_buf(df_train_test, "Dados Originais (Total)",        endog_cols))
pdf.add_image_buf(constellation_buf(fc_total,      f"Forecast (Total)\n{param_str}", endog_cols))
pdf.add_comparison_buf(comparison_buf(df_train_test, fc_total, "Dados Originais (Total)", f"Forecast (Total)\n{param_str}", endog_cols))
pdf.add_image_buf(magnitude_buf(df_train_test, fc_total, f"Mapa de Magnitude (Total)\n{param_str}", endog_cols))

pdf.output(report_name)
print(f"PDF gerado: {report_name}")

/tmp/ipykernel_63366/1453607665.py:13: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  self.cell(0, 10, title, ln=True, align="C")
/tmp/ipykernel_63366/1453607665.py:25: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  self.cell(0, 8, title, ln=True)
/tmp/ipykernel_63366/1453607665.py:18: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=False use new_x=XPos.RIGHT, new_y=YPos.TOP.
  self.cell(70, 6, f"  {label}", ln=False)
/tmp/ipykernel_63366/1453607665.py:20: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  self.cell(0, 6, f": {value}", ln=True)
/tmp/ipykernel_63366/3071849062.py:16: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.ce

PDF gerado: /home/mgs/Documents/FACOM/Projects/projeto-dpd/masters-varmax/reports/relatorio-varmax-p1-xl1-r1-i2(original).pdf
